# RAG Evaluation Metrics

## 1. Retrieval Evaluation (The "Input" Phase)
These metrics measure the quality of the information retrieved by your vector database or search system before it is passed to the Large Language Model (LLM).
### Context Precision
Measures whether the relevant chunks are ranked higher than irrelevant ones.
- A higher score indicates that the retriever is placing the most useful and relevant information at the top of the retrieved context.
- A lower score suggests that irrelevant or noisy chunks are being retrieved earlier than useful ones.
### Context Recall
Measures the retriever's ability to fetch all the necessary information required to answer the user's query.
- A higher score means the retriever successfully captured most or all of the important information.
- A lower score indicates that critical facts or supporting context are missing from the retrieved documents.

## 2. Generation Evaluation (The "Output" Phase)
These metrics evaluate how accurately and appropriately the LLM generates answers using the retrieved context.
### Faithfulness
Measures the factual consistency of the generated response with respect to the retrieved context.
- The model checks whether every claim made in the answer can be logically inferred from the provided context.
- A lower score indicates hallucinations, unsupported claims, or the introduction of external knowledge not present in the retrieved documents.
### Answer Relevancy
Measures how relevant, focused, and complete the generated answer is with respect to the user's question.
- It evaluates whether the response directly addresses the query.
- It penalizes:
  - incomplete answers
  - off-topic responses
  - redundant or unnecessary information
  - vague explanations that avoid the main question

In [1]:
from dotenv import load_dotenv
import os
import math
import os
import re
import textwrap
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

from openai import AsyncOpenAI


# from ragas.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from ragas.llms import llm_factory
# from ragas.metrics.collections import answer_relevancy, context_precision, context_recall, faithfulness
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import EvaluationDataset, SingleTurnSample, evaluate


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

load_dotenv()

/var/folders/9k/9w2ktp1j4m1d3sj_kdjsll7r0000gq/T/ipykernel_79191/560397546.py:21: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/9k/9w2ktp1j4m1d3sj_kdjsll7r0000gq/T/ipykernel_79191/560397546.py:21: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/9k/9w2ktp1j4m1d3sj_kdjsll7r0000gq/T/ipykernel_79191/560397546.py:21: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/

True

In [2]:
client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# LLM
llm = llm_factory(
    "openai/gpt-4o",
    client=client
)

# Embeddings
# embeddings = OpenAIEmbeddings(
#     model="openai/text-embedding-3-small",
#     client=client
# )

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

def show_df(df: pd.DataFrame, title: Optional[str] = None) -> pd.DataFrame:
    """Display a dataframe with an optional title and return it for further use."""
    if title:
        print(title)
    else:
        display(df)
    return df

In [4]:
# A small manual dataset using the older field names that many tutorials still show.
samples = [
    {
        "question": "What is RAGAS used for?",
        "answer": "RAGAS is used to evaluate RAG pipelines using metrics such as faithfulness and context recall.",
        "contexts": [
            "RAGAS is a framework for evaluating Retrieval-Augmented Generation systems.",
            "It includes metrics for faithfulness, answer relevance, context precision, and context recall.",
        ],
        "ground_truth": "RAGAS is used to evaluate Retrieval-Augmented Generation systems with metrics for retrieval and generation quality.",
    },
    {
        "question": "Why does context recall matter in RAG?",
        "answer": "Context recall matters because it checks whether the retriever found the information needed to answer the question.",
        "contexts": [
            "Context recall measures whether required information from the reference answer appears in the retrieved contexts.",
            "Faithfulness checks whether generated answer claims are supported by retrieved context.",
        ],
        "ground_truth": "Context recall matters because a RAG system cannot answer reliably if important supporting information is not retrieved.",
    },
    {
        "question": "What does low faithfulness suggest?",
        "answer": "Low faithfulness suggests the answer may contain claims that are not supported by the retrieved context.",
        "contexts": [
            "Faithfulness measures whether claims in the response can be inferred from the retrieved context.",
        ],
        "ground_truth": "Low faithfulness suggests possible hallucination or unsupported claims in the generated answer.",
    },
]

legacy_df = pd.DataFrame(samples)
show_df(legacy_df, "Legacy-style evaluation records")

Legacy-style evaluation records


,question,answer,contexts,ground_truth
0,What is RAGAS used for?,RAGAS is used to evaluate RAG pipelines using metrics such as faithfulness and context recall.,"[RAGAS is a framework for evaluating Retrieval-Augmented Generation systems., It includes metrics for faithfulness, ...",RAGAS is used to evaluate Retrieval-Augmented Generation systems with metrics for retrieval and generation quality.
1,Why does context recall matter in RAG?,Context recall matters because it checks whether the retriever found the information needed to answer the question.,"[Context recall measures whether required information from the reference answer appears in the retrieved contexts., ...",Context recall matters because a RAG system cannot answer reliably if important supporting information is not retrie...
2,What does low faithfulness suggest?,Low faithfulness suggests the answer may contain claims that are not supported by the retrieved context.,[Faithfulness measures whether claims in the response can be inferred from the retrieved context.],Low faithfulness suggests possible hallucination or unsupported claims in the generated answer.


In [5]:
RAGAS_FIELD_MAP = {
    "question": "user_input",
    "answer": "response",
    "contexts": "retrieved_contexts",
    "ground_truth": "reference",
}

def convert_legacy_to_ragas(df: pd.DataFrame) -> pd.DataFrame:
    """Convert a dataframe with legacy field names to RAGAS field names."""
    return df.rename(columns=RAGAS_FIELD_MAP)

converted_df = convert_legacy_to_ragas(legacy_df)
show_df(converted_df, "Converted to RAGAS-style evaluation records")

Converted to RAGAS-style evaluation records


,user_input,response,retrieved_contexts,reference
0,What is RAGAS used for?,RAGAS is used to evaluate RAG pipelines using metrics such as faithfulness and context recall.,"[RAGAS is a framework for evaluating Retrieval-Augmented Generation systems., It includes metrics for faithfulness, ...",RAGAS is used to evaluate Retrieval-Augmented Generation systems with metrics for retrieval and generation quality.
1,Why does context recall matter in RAG?,Context recall matters because it checks whether the retriever found the information needed to answer the question.,"[Context recall measures whether required information from the reference answer appears in the retrieved contexts., ...",Context recall matters because a RAG system cannot answer reliably if important supporting information is not retrie...
2,What does low faithfulness suggest?,Low faithfulness suggests the answer may contain claims that are not supported by the retrieved context.,[Faithfulness measures whether claims in the response can be inferred from the retrieved context.],Low faithfulness suggests possible hallucination or unsupported claims in the generated answer.


##  RAGAS 
 - EvaluationDataset : EvaluationDataset is essential because it provides the standardized input required to measure how effectively your RAG system retrieves data and generates answers. It acts as the baseline for the "LLM-as-a-judge"
 - SingleTurnSample : represents one complete question-and-answer interaction from a RAG pipeline. It serves as the basic, single-row building block used to structure, validate, and evaluate an individual query before scaling up to a full dataset.

In [6]:
samples = [
    SingleTurnSample(
        user_input=record.get("user_input", ""),
        response=record.get("response", ""),
        retrieved_contexts=record.get("retrieved_contexts", []),
        reference=record.get("reference", ""),
    )
    for record in converted_df.to_dict(orient="records")
]

In [7]:
samples

[SingleTurnSample(user_input='What is RAGAS used for?', retrieved_contexts=['RAGAS is a framework for evaluating Retrieval-Augmented Generation systems.', 'It includes metrics for faithfulness, answer relevance, context precision, and context recall.'], reference_contexts=None, retrieved_context_ids=None, reference_context_ids=None, response='RAGAS is used to evaluate RAG pipelines using metrics such as faithfulness and context recall.', multi_responses=None, reference='RAGAS is used to evaluate Retrieval-Augmented Generation systems with metrics for retrieval and generation quality.', rubrics=None, persona_name=None, query_style=None, query_length=None),
 SingleTurnSample(user_input='Why does context recall matter in RAG?', retrieved_contexts=['Context recall measures whether required information from the reference answer appears in the retrieved contexts.', 'Faithfulness checks whether generated answer claims are supported by retrieved context.'], reference_contexts=None, retrieved_c

In [8]:
eval_dataset = EvaluationDataset(samples)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=3)

In [9]:
eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,What is RAGAS used for?,"[RAGAS is a framework for evaluating Retrieval-Augmented Generation systems., It includes metrics for faithfulness, ...",RAGAS is used to evaluate RAG pipelines using metrics such as faithfulness and context recall.,RAGAS is used to evaluate Retrieval-Augmented Generation systems with metrics for retrieval and generation quality.
1,Why does context recall matter in RAG?,"[Context recall measures whether required information from the reference answer appears in the retrieved contexts., ...",Context recall matters because it checks whether the retriever found the information needed to answer the question.,Context recall matters because a RAG system cannot answer reliably if important supporting information is not retrie...
2,What does low faithfulness suggest?,[Faithfulness measures whether claims in the response can be inferred from the retrieved context.],Low faithfulness suggests the answer may contain claims that are not supported by the retrieved context.,Low faithfulness suggests possible hallucination or unsupported claims in the generated answer.


In [10]:


results = evaluate(
    eval_dataset,
    llm=llm,
    embeddings=embeddings,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


In [11]:
results

{'faithfulness': 0.5556, 'answer_relevancy': 0.8871, 'context_precision': 1.0000, 'context_recall': 0.6667}